In [10]:
from pathlib import Path

import hist
import matplotlib.pyplot as plt
import mplhep as mh
import numpy as np
import pandas as pd
import uproot

from utils import TREE_NAME, branch_summary, branch_values, hist_bins, safe_name, sample_label

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 240)
mh.style.use(mh.styles.CMS)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = PROJECT_ROOT / "output" / "florian"
PLOTS_ROOT = PROJECT_ROOT / "plots" / "florian"
INPUT_FILES = [INPUT_DIR / "ZKK.root", INPUT_DIR / "Zmumu.root", INPUT_DIR / "Zpipi.root"]

SDST_PREFIX = "SDST_"
RAW_SDST_PREFIX = "RAWSDST_"
RAW_FADANA_PREFIX = "RAWFADANA_"

files = [Path(path) for path in INPUT_FILES]
missing = [path for path in files if not path.exists()]
assert not missing, "Missing input ROOT files:\n" + "\n".join(str(path) for path in missing)

PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Found {len(files)} ROOT files")
print(f"Writing plots and tables under {PLOTS_ROOT}")


Found 3 ROOT files
Writing plots and tables under /eos/home-j/joshin/workspace-eos/delphi/delphi-nanoaod/plots/florian


In [11]:
def select_branches(paths: list[Path], prefix: str | None = None) -> list[str]:
    branches = set()
    for path in paths:
        with uproot.open(path) as root_file:
            branches.update(
                name for name in root_file[TREE_NAME].keys()
                if prefix is None or name.startswith(prefix)
            )
    return sorted(branches)


all_branches = select_branches(files)
pd.DataFrame({"branch": all_branches}).to_csv(PLOTS_ROOT / "all_branches.csv", index=False)
print(f"Found {len(all_branches)} branches across {len(files)} ROOT files")
display(pd.DataFrame({"branch": all_branches}))


Found 523 branches across 3 ROOT files


,branch
0,Event_eventNumber
1,Event_runNumber
2,RAWFADANA_ElidRaw_gammaConvTag
3,RAWFADANA_ElidRaw_paIdx
4,RAWFADANA_ElidRaw_refitMomentum
...,...
518,SDST_nV0
519,SDST_nV0Hyp
520,SDST_nVdHit
521,SDST_nVdUnHit


In [12]:
GROUP_ALIASES = {"TracRaw": "TrackRaw"}


def variable_name(branch: str, prefix: str) -> str:
    return branch[len(prefix):] if branch.startswith(prefix) else branch


def group_name(branch: str, prefix: str) -> str:
    group = variable_name(branch, prefix).split("_", 1)[0]
    if group.startswith("n") and len(group) > 1 and group[1].isupper():
        group = group[1:]
    return GROUP_ALIASES.get(group, group)


def plot_branch(branch: str, samples: list[tuple[str, object, set[str], dict[str, str]]], output_dir: Path, prefix: str) -> dict:
    values_by_sample = {}
    for label, tree, keys, typenames in samples:
        if branch not in keys:
            continue
        values, _ = branch_values(tree, branch, typenames)
        if values.size:
            values_by_sample[label] = values

    if not values_by_sample:
        return {"branch": branch, "status": "empty", "entries": 0, "path": ""}

    values = np.concatenate(list(values_by_sample.values()))
    if np.all(values == 0):
        return {"branch": branch, "status": "all_zero", "entries": int(values.size), "path": ""}

    variable = variable_name(branch, prefix)
    nbins, lo, hi = hist_bins(values)
    fig, ax = plt.subplots(figsize=(12, 9))

    for label, sample_values in values_by_sample.items():
        h = hist.Hist.new.Reg(nbins, lo, hi, name="value", label=variable).Double()
        h.fill(value=sample_values)
        mh.histplot(h, ax=ax, histtype="step", label=label, yerr=True)

    ax.set_xlabel(variable)
    ax.set_ylabel("Entries")
    ax.legend()
    mh.label.exp_label(
        exp="DELPHI",
        text="Private Work",
        rlabel=r"LEP1, $\sqrt{s}\sim 91.25$ GeV",
        loc=0,
        ax=ax,
    )
    fig.tight_layout()

    path = output_dir / group_name(branch, prefix) / f"{safe_name(variable)}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)

    return {
        "branch": branch,
        "status": "plotted",
        "entries": int(values.size),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "path": str(path),
    }


def plot_part(part: str, prefix: str) -> dict:
    output_dir = PLOTS_ROOT / part
    output_dir.mkdir(parents=True, exist_ok=True)
    for png in output_dir.rglob("*.png"):
        png.unlink()

    branches = select_branches(files, prefix)
    samples = []
    for path in files:
        tree = uproot.open(path)[TREE_NAME]
        samples.append((sample_label(path), tree, set(tree.keys()), tree.typenames()))

    branch_table = branch_summary(files, branches)
    manifest = pd.DataFrame(plot_branch(branch, samples, output_dir, prefix) for branch in branches)
    summary = pd.DataFrame([
        {
            "part": part,
            "branches": len(branches),
            "plotted": int((manifest["status"] == "plotted").sum()),
            "all_zero": int((manifest["status"] == "all_zero").sum()),
            "empty": int((manifest["status"] == "empty").sum()),
            "output_dir": str(output_dir),
        }
    ])

    branch_table.to_csv(output_dir / "branch_summary.csv", index=False)
    manifest.to_csv(output_dir / "plot_manifest.csv", index=False)
    summary.to_csv(output_dir / "plot_variables_summary.csv", index=False)

    row = summary.iloc[0]
    print(f"[{part}] plotted {row.plotted} / {row.branches} branches; skipped {row.all_zero} all-zero and {row.empty} empty branches")
    return {"branches": branches, "branch_table": branch_table, "manifest": manifest, "summary": summary}


In [16]:
plot_sdst = plot_part("sdst", SDST_PREFIX)
plot_raw_sdst = plot_part("raw_sdst", RAW_SDST_PREFIX)
plot_raw_fadana = plot_part("raw_fadana", RAW_FADANA_PREFIX)

plot_summaries = pd.concat(
    [plot_sdst["summary"], plot_raw_sdst["summary"], plot_rawfadana["summary"]],
    ignore_index=True,
)
plot_summaries.to_csv(PLOTS_ROOT / "plot_variables_summary.csv", index=False)
display(plot_summaries)


[sdst] plotted 225 / 273 branches; skipped 29 all-zero and False empty branches
[raw_sdst] plotted 104 / 124 branches; skipped 4 all-zero and False empty branches
[raw_fadana] plotted 71 / 124 branches; skipped 10 all-zero and False empty branches


,part,branches,plotted,all_zero,empty,output_dir
0,sdst,273,225,29,19,/eos/home-j/joshin/workspace-eos/delphi/delphi...
1,raw_sdst,124,104,4,16,/eos/home-j/joshin/workspace-eos/delphi/delphi...
2,rawfadana,124,71,10,43,/eos/home-j/joshin/workspace-eos/delphi/delphi...
